# Lab 07: Object Detection and Semantic Segmentation

            **Duration:** 3 hours  
            **Lecture alignment:** Week 7 — Detection and segmentation  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Represent class, bounding-box, and mask targets correctly.
- Train a compact multi-task vision model.
- Evaluate detection and segmentation with class accuracy, box IoU, mask IoU, and Dice.

            ## Three-hour activity plan

            - 0–30 min: target formats and synthetic data
- 30–65 min: encoder, heads, and multi-task losses
- 65–125 min: train and debug
- 125–160 min: IoU/Dice evaluation and overlays
- 160–180 min: error analysis and checks


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20267
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_07")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_07"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 7, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Which task head—classification, box regression, or pixel segmentation—will learn fastest on this dataset, and what metric will reveal it?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Generate images with class, box, and mask targets


In [ ]:
def make_shapes(n,size=32):
    images=.04*torch.randn(n,1,size,size); masks=torch.zeros(n,1,size,size)
    boxes=torch.zeros(n,4); labels=torch.randint(0,2,(n,)); yy,xx=torch.meshgrid(torch.arange(size),torch.arange(size),indexing="ij")
    for i,label in enumerate(labels.tolist()):
        radius=int(torch.randint(4,8,(1,))); cx=int(torch.randint(radius+1,size-radius-1,(1,))); cy=int(torch.randint(radius+1,size-radius-1,(1,)))
        if label==0: mask=(xx-cx).abs().le(radius)&(yy-cy).abs().le(radius)
        else: mask=(xx-cx).square()+(yy-cy).square()<=radius**2
        masks[i,0]=mask.float(); images[i,0]+=masks[i,0]*(.75+.2*torch.rand(1))
        ys,xs=torch.where(mask); boxes[i]=torch.tensor([xs.min(),ys.min(),xs.max(),ys.max()])/float(size-1)
    return images.clamp(0,1),masks,boxes,labels
X,masks,boxes,labels=make_shapes(220 if FAST_MODE else 1000)
split=int(.8*len(X)); train_data=(X[:split],masks[:split],boxes[:split],labels[:split]); test_data=(X[split:],masks[split:],boxes[split:],labels[split:])
print({"image":tuple(X.shape),"mask":tuple(masks.shape),"boxes_range":(boxes.min().item(),boxes.max().item())})


## Activity 2 — Train a multi-task detector/segmenter


In [ ]:
class MultiTaskVision(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.decoder=nn.Sequential(nn.ConvTranspose2d(16,8,4,stride=2,padding=1),nn.ReLU(),nn.ConvTranspose2d(8,1,4,stride=2,padding=1))
        self.shared=nn.Sequential(nn.Flatten(),nn.Linear(16*8*8,32),nn.ReLU())
        self.box_head=nn.Linear(32,4); self.class_head=nn.Linear(32,2)
    def forward(self,x):
        z=self.encoder(x); h=self.shared(z)
        return self.decoder(z),torch.sigmoid(self.box_head(h)),self.class_head(h)
model=MultiTaskVision().to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=.006)
loader=DataLoader(TensorDataset(*train_data),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
history=[]
for _ in range(7 if FAST_MODE else 25):
    model.train(); total=0
    for xb,mb,bb,yb in loader:
        xb,mb,bb,yb=[v.to(DEVICE) for v in (xb,mb,bb,yb)]; opt.zero_grad(); pm,pb,pc=model(xb)
        loss=F.binary_cross_entropy_with_logits(pm,mb)+2*F.smooth_l1_loss(pb,bb)+.5*F.cross_entropy(pc,yb)
        loss.backward(); opt.step(); total+=loss.item()*len(xb)
    history.append(total/len(train_data[0]))


## Activity 3 — Dice, mask IoU, box IoU, and overlays


In [ ]:
def box_iou(a,b):
    x0=torch.maximum(a[:,0],b[:,0]); y0=torch.maximum(a[:,1],b[:,1]); x1=torch.minimum(a[:,2],b[:,2]); y1=torch.minimum(a[:,3],b[:,3])
    inter=(x1-x0).clamp(min=0)*(y1-y0).clamp(min=0)
    area_a=(a[:,2]-a[:,0]).clamp(min=0)*(a[:,3]-a[:,1]).clamp(min=0); area_b=(b[:,2]-b[:,0]).clamp(min=0)*(b[:,3]-b[:,1]).clamp(min=0)
    return inter/(area_a+area_b-inter+1e-8)
model.eval(); Xt,mt,bt,yt=test_data
with torch.no_grad(): pm,pb,pc=model(Xt.to(DEVICE)); probs=pm.sigmoid().cpu(); pb=pb.cpu(); pc=pc.cpu()
pred_masks=(probs>.5).float(); intersection=(pred_masks*mt).sum((1,2,3)); union=((pred_masks+mt)>0).float().sum((1,2,3))
dice=(2*intersection/(pred_masks.sum((1,2,3))+mt.sum((1,2,3))+1e-8)).mean().item(); mask_iou=(intersection/(union+1e-8)).mean().item()
mean_box_iou=box_iou(pb,bt).mean().item(); class_acc=(pc.argmax(1)==yt).float().mean().item()
metrics={"dice":dice,"mask_iou":mask_iou,"box_iou":mean_box_iou,"class_accuracy":class_acc,"final_loss":history[-1]}
print(json.dumps(metrics,indent=2)); (ARTIFACT_DIR/"metrics.json").write_text(json.dumps(metrics,indent=2))
fig,axes=plt.subplots(3,4,figsize=(9,7))
for j in range(4):
    axes[0,j].imshow(Xt[j,0],cmap="gray"); axes[1,j].imshow(mt[j,0],cmap="gray"); axes[2,j].imshow(probs[j,0],cmap="magma",vmin=0,vmax=1)
    for ax in axes[:,j]: ax.axis("off")
axes[0,0].set_ylabel("image");axes[1,0].set_ylabel("target");axes[2,0].set_ylabel("prediction")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"detection_segmentation.png",dpi=150);plt.show()


## Automated checks


In [ ]:
assert pm.shape==mt.shape and pb.shape==(len(Xt),4) and pc.shape==(len(Xt),2)
assert all(math.isfinite(v) for v in metrics.values())
assert 0<=dice<=1 and 0<=mean_box_iou<=1 and boxes.min()>=0 and boxes.max()<=1
assert history[-1]<history[0]
print("All Lab 07 checks passed.")


## Deliverables

                - Verified target tensors
- Executable multi-task model
- Metrics JSON and qualitative overlay grid
- Short comparison of classification, detection, and segmentation outputs

                Submit the executed notebook and the files created in `/content/artifacts/lab_07/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: generate multiple objects per image and add a background/foreground imbalance treatment.")
else:
    print("Extension disabled: multi-object images, non-maximum suppression, and focal loss.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
